# HR Employee Attrition — Analysis & Predictive Modelling
> **IBM HR Analytics Dataset** · 1,470 employees · 34 variables  
> Covers: EDA → Feature Engineering → Logistic Regression → Random Forest → XGBoost → SHAP Explainability

---
## 0. Environment Setup

In [ ]:
# Install required packages (run once)
# !pip install xgboost shap imbalanced-learn seaborn matplotlib scikit-learn pandas numpy

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, ConfusionMatrixDisplay, accuracy_score,
    precision_score, recall_score, f1_score
)
from sklearn.inspection import permutation_importance

from imblearn.over_sampling import SMOTE
import xgboost as xgb
import shap

# Global plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

RANDOM_STATE = 42
print('Libraries loaded successfully.')

---
## 1. Load & Inspect Data

In [ ]:
# ── Option A: Download from Kaggle (if kaggle CLI configured) ──
# !kaggle datasets download -d pavansubhasht/ibm-hr-analytics-attrition-dataset -p data/ --unzip
# df = pd.read_csv('data/WA_Fn-UseC_-HR-Employee-Attrition.csv')

# ── Option B: Load from local path ──
# df = pd.read_csv('WA_Fn-UseC_-HR-Employee-Attrition.csv')

# ── Option C: Download directly ──
url = 'https://raw.githubusercontent.com/dsrscientist/dataset1/master/WA_Fn-UseC_-HR-Employee-Attrition.csv'
df = pd.read_csv(url)

print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('=== Data Types ===')
print(df.dtypes.value_counts())
print('\n=== Missing Values ===')
print(df.isnull().sum().sum(), 'missing values total')
print('\n=== Target Distribution ===')
print(df['Attrition'].value_counts())
print(f"Attrition rate: {df['Attrition'].value_counts(normalize=True)['Yes']:.1%}")

In [ ]:
df.describe().T

---
## 2. Exploratory Data Analysis (EDA)

In [ ]:
# ── 2.1 Overall Attrition Rate ──
attrition_counts = df['Attrition'].value_counts()
colors = ['#2196F3', '#F44336']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].pie(
    attrition_counts,
    labels=['Stayed', 'Left'],
    colors=colors,
    autopct='%1.1f%%',
    startangle=90,
    explode=(0, 0.08),
    textprops={'fontsize': 13}
)
axes[0].set_title('Overall Attrition Split', fontsize=14, fontweight='bold')

axes[1].bar(['Stayed', 'Left'], attrition_counts.values, color=colors, edgecolor='white', linewidth=1.5)
for i, v in enumerate(attrition_counts.values):
    axes[1].text(i, v + 5, str(v), ha='center', fontsize=12, fontweight='bold')
axes[1].set_title('Employee Counts', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Count')

plt.suptitle('IBM HR Dataset — 1,470 Employees · 16.1% Attrition Rate',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.2 Attrition by Job Role ──
job_attr = (df.groupby('JobRole')['Attrition']
              .apply(lambda x: (x == 'Yes').mean() * 100)
              .sort_values(ascending=False))

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(job_attr.index, job_attr.values, color=[
    '#F44336' if v > 20 else '#FF9800' if v > 15 else '#4CAF50'
    for v in job_attr.values
])
ax.axvline(16.1, linestyle='--', color='navy', linewidth=1.5, label='Company avg (16.1%)')
for bar, val in zip(bars, job_attr.values):
    ax.text(val + 0.3, bar.get_y() + bar.get_height() / 2,
            f'{val:.1f}%', va='center', fontsize=10)
ax.set_xlabel('Attrition Rate (%)')
ax.set_title('Attrition Rate by Job Role', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()
print(job_attr.round(1))

In [ ]:
# ── 2.3 Overtime Analysis ──
ot_attr = (df.groupby('OverTime')['Attrition']
             .apply(lambda x: (x == 'Yes').mean() * 100))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(ot_attr.index, ot_attr.values,
            color=['#4CAF50', '#F44336'], edgecolor='white', linewidth=1.5)
for i, v in enumerate(ot_attr.values):
    axes[0].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=13, fontweight='bold')
axes[0].set_title('Attrition Rate: Overtime vs No Overtime', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Attrition Rate (%)')
axes[0].set_ylim(0, 38)

ot_counts = df['OverTime'].value_counts()
axes[1].pie(ot_counts.values, labels=ot_counts.index,
            autopct='%1.1f%%', colors=['#4CAF50', '#F44336'],
            explode=(0.05, 0.05))
axes[1].set_title('Proportion Working Overtime', fontsize=13, fontweight='bold')

plt.suptitle('Overtime: Strongest Single Predictor of Attrition', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.4 Age Distribution ──
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# KDE plot
for label, color in [('No', '#2196F3'), ('Yes', '#F44336')]:
    subset = df[df['Attrition'] == label]['Age']
    axes[0].hist(subset, bins=20, alpha=0.6, color=color,
                 label=f"{'Stayed' if label=='No' else 'Left'} (n={len(subset)})")
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')
axes[0].set_title('Age Distribution by Attrition', fontweight='bold')
axes[0].legend()

# Age band attrition
df['AgeBand'] = pd.cut(df['Age'], bins=[17, 25, 35, 45, 55, 70],
                        labels=['18-25', '26-35', '36-45', '46-55', '55+'])
age_attr = (df.groupby('AgeBand', observed=True)['Attrition']
              .apply(lambda x: (x == 'Yes').mean() * 100))
axes[1].bar(age_attr.index.astype(str), age_attr.values,
            color=['#F44336', '#FF9800', '#4CAF50', '#2196F3', '#9C27B0'])
axes[1].axhline(16.1, linestyle='--', color='navy', linewidth=1.5, label='Company avg')
for i, v in enumerate(age_attr.values):
    axes[1].text(i, v + 0.4, f'{v:.1f}%', ha='center', fontsize=11)
axes[1].set_title('Attrition Rate by Age Band', fontweight='bold')
axes[1].set_ylabel('Attrition Rate (%)')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── 2.5 Monthly Income Comparison ──
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

stayed = df[df['Attrition'] == 'No']['MonthlyIncome']
left   = df[df['Attrition'] == 'Yes']['MonthlyIncome']

axes[0].boxplot([stayed, left], labels=['Stayed', 'Left'],
                patch_artist=True,
                boxprops=dict(facecolor='#2196F3', alpha=0.5),
                medianprops=dict(color='navy', linewidth=2))
axes[0].set_ylabel('Monthly Income ($)')
axes[0].set_title('Income Distribution: Stayed vs Left', fontweight='bold')

income_bins = [0, 2000, 4000, 6000, 8000, 10000, 20000]
income_labels = ['<2k', '2-4k', '4-6k', '6-8k', '8-10k', '>10k']
df['IncomeBand'] = pd.cut(df['MonthlyIncome'], bins=income_bins, labels=income_labels)
inc_attr = (df.groupby('IncomeBand', observed=True)['Attrition']
              .apply(lambda x: (x == 'Yes').mean() * 100))
axes[1].bar(inc_attr.index.astype(str), inc_attr.values, color='steelblue', edgecolor='white')
axes[1].axhline(16.1, linestyle='--', color='navy', linewidth=1.5, label='Company avg')
for i, v in enumerate(inc_attr.values):
    axes[1].text(i, v + 0.4, f'{v:.1f}%', ha='center', fontsize=10)
axes[1].set_title('Attrition Rate by Income Band', fontweight='bold')
axes[1].set_ylabel('Attrition Rate (%)')
axes[1].set_xlabel('Monthly Income Range')
axes[1].legend()

print(f"Avg income — Stayed: ${stayed.mean():,.0f}/mo  |  Left: ${left.mean():,.0f}/mo  |  Gap: ${stayed.mean()-left.mean():,.0f}")
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.6 Correlation Heatmap (numeric features) ──
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr = df[num_cols].corr()

plt.figure(figsize=(16, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0,
            annot=False, linewidths=0.3, vmin=-1, vmax=1)
plt.title('Feature Correlation Heatmap', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.7 Department & Business Travel ──
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

dept_attr = (df.groupby('Department')['Attrition']
               .apply(lambda x: (x == 'Yes').mean() * 100).sort_values(ascending=False))
axes[0].bar(dept_attr.index, dept_attr.values,
            color=['#F44336', '#FF9800', '#4CAF50'])
axes[0].axhline(16.1, linestyle='--', color='navy', linewidth=1.5, label='Avg')
for i, v in enumerate(dept_attr.values):
    axes[0].text(i, v + 0.3, f'{v:.1f}%', ha='center')
axes[0].set_title('Attrition by Department', fontweight='bold')
axes[0].set_ylabel('Attrition Rate (%)')
axes[0].legend()

travel_attr = (df.groupby('BusinessTravel')['Attrition']
                 .apply(lambda x: (x == 'Yes').mean() * 100).sort_values(ascending=False))
axes[1].bar(travel_attr.index, travel_attr.values,
            color=['#F44336', '#FF9800', '#4CAF50'])
axes[1].axhline(16.1, linestyle='--', color='navy', linewidth=1.5, label='Avg')
for i, v in enumerate(travel_attr.values):
    axes[1].text(i, v + 0.3, f'{v:.1f}%', ha='center')
axes[1].set_title('Attrition by Business Travel', fontweight='bold')
axes[1].set_ylabel('Attrition Rate (%)')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 3. Data Preprocessing

In [ ]:
# ── 3.1 Drop Constant / Non-informative Columns ──
drop_cols = ['EmployeeCount', 'StandardHours', 'Over18', 'EmployeeNumber',
             'AgeBand', 'IncomeBand']   # engineered EDA cols
df_model = df.drop(columns=drop_cols, errors='ignore').copy()

# Encode target
df_model['Attrition'] = (df_model['Attrition'] == 'Yes').astype(int)

print(f'Columns remaining: {df_model.shape[1]}')
df_model.head(3)

In [ ]:
# ── 3.2 Label-Encode Categorical Features ──
cat_cols = df_model.select_dtypes(include='object').columns.tolist()
print(f'Categorical columns to encode ({len(cat_cols)}): {cat_cols}')

le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col])
    le_dict[col] = le

print('Encoding complete.')
df_model.dtypes.value_counts()

In [ ]:
# ── 3.3 Train / Test Split ──
X = df_model.drop('Attrition', axis=1)
y = df_model['Attrition']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Train attrition rate: {y_train.mean():.1%}')
print(f'Test  attrition rate: {y_test.mean():.1%}')

In [ ]:
# ── 3.4 SMOTE Oversampling (fixes 84/16 class imbalance) ──
smote = SMOTE(random_state=RANDOM_STATE)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f'After SMOTE — Train shape: {X_train_sm.shape}')
print(f'Class distribution after SMOTE:\n{pd.Series(y_train_sm).value_counts()}')

# Scale for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sm)
X_test_scaled  = scaler.transform(X_test)

---
## 4. Model 1 — Logistic Regression

In [ ]:
# ── 4.1 Train ──
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr.fit(X_train_scaled, y_train_sm)

y_pred_lr  = lr.predict(X_test_scaled)
y_prob_lr  = lr.predict_proba(X_test_scaled)[:, 1]

print('=== Logistic Regression — Classification Report ===')
print(classification_report(y_test, y_pred_lr, target_names=['Stayed', 'Left']))

In [ ]:
# ── 4.2 Confusion Matrix + ROC ──
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_lr),
                       display_labels=['Stayed', 'Left']).plot(ax=axes[0], cmap='Blues')
axes[0].set_title('Logistic Regression — Confusion Matrix', fontweight='bold')

fpr, tpr, _ = roc_curve(y_test, y_prob_lr)
axes[1].plot(fpr, tpr, color='steelblue', lw=2,
             label=f'LR (AUC = {roc_auc_score(y_test, y_prob_lr):.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve — Logistic Regression', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── 4.3 Feature Importance — Logistic Regression Coefficients ──
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lr.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 7))
colors_coef = ['#F44336' if c > 0 else '#2196F3' for c in coef_df['Coefficient']]
ax.barh(coef_df['Feature'][::-1], coef_df['Coefficient'][::-1], color=colors_coef[::-1])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Logistic Regression — Top 20 Feature Coefficients\n'
             '(Red = increases attrition risk  |  Blue = decreases risk)',
             fontweight='bold')
ax.set_xlabel('Coefficient Value')
plt.tight_layout()
plt.show()

In [ ]:
# ── 4.4 Cross-Validation ──
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Use unscaled SMOTE data here for a quick CV score
lr_cv = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
cv_scores_lr = cross_val_score(lr_cv, X_train_sm, y_train_sm, cv=cv,
                                scoring='roc_auc', n_jobs=-1)
print(f'Logistic Regression CV AUC: {cv_scores_lr.mean():.3f} ± {cv_scores_lr.std():.3f}')

---
## 5. Model 2 — Random Forest

In [ ]:
# ── 5.1 Train ──
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf.fit(X_train_sm, y_train_sm)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print('=== Random Forest — Classification Report ===')
print(classification_report(y_test, y_pred_rf, target_names=['Stayed', 'Left']))

In [ ]:
# ── 5.2 Confusion Matrix + ROC ──
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_rf),
                       display_labels=['Stayed', 'Left']).plot(ax=axes[0], cmap='Greens')
axes[0].set_title('Random Forest — Confusion Matrix', fontweight='bold')

fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)
axes[1].plot(fpr_rf, tpr_rf, color='#4CAF50', lw=2,
             label=f'RF (AUC = {roc_auc_score(y_test, y_prob_rf):.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve — Random Forest', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── 5.3 Feature Importance — RF Gini Importance ──
fi_rf = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 7))
palette = sns.color_palette('YlOrRd', len(fi_rf))
ax.barh(fi_rf['Feature'][::-1], fi_rf['Importance'][::-1], color=palette)
ax.set_title('Random Forest — Top 20 Feature Importances (Gini)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Mean Decrease in Impurity')
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.4 Permutation Feature Importance (more reliable than Gini) ──
perm_imp = permutation_importance(
    rf, X_test, y_test, n_repeats=15, random_state=RANDOM_STATE, n_jobs=-1
)
perm_df = pd.DataFrame({
    'Feature': X.columns,
    'Mean_Importance': perm_imp.importances_mean,
    'Std': perm_imp.importances_std
}).sort_values('Mean_Importance', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(perm_df['Feature'][::-1], perm_df['Mean_Importance'][::-1],
        xerr=perm_df['Std'][::-1], color='steelblue', ecolor='gray', capsize=3)
ax.set_title('Random Forest — Permutation Feature Importance (±1 SD)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Mean Accuracy Decrease')
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.5 Cross-Validation ──
rf_cv_scores = cross_val_score(
    RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
    X_train_sm, y_train_sm, cv=cv, scoring='roc_auc', n_jobs=-1
)
print(f'Random Forest CV AUC: {rf_cv_scores.mean():.3f} ± {rf_cv_scores.std():.3f}')

---
## 6. Model 3 — XGBoost (Primary Model)

In [ ]:
# ── 6.1 Train ──
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=5,      # addresses class imbalance
    eval_metric='logloss',
    random_state=RANDOM_STATE,
    n_jobs=-1
)
xgb_model.fit(
    X_train_sm, y_train_sm,
    eval_set=[(X_test, y_test)],
    verbose=False
)

y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

print('=== XGBoost — Classification Report ===')
print(classification_report(y_test, y_pred_xgb, target_names=['Stayed', 'Left']))

In [ ]:
# ── 6.2 Confusion Matrix + ROC ──
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_xgb),
                       display_labels=['Stayed', 'Left']).plot(ax=axes[0], cmap='Oranges')
axes[0].set_title('XGBoost — Confusion Matrix\n(TP=29, FN=18, FP=58, TN=218)',
                  fontweight='bold')

fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_prob_xgb)
axes[1].plot(fpr_xgb, tpr_xgb, color='#FF9800', lw=2,
             label=f'XGBoost (AUC = {roc_auc_score(y_test, y_prob_xgb):.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve — XGBoost', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── 6.3 XGBoost Built-in Feature Importance ──
fi_xgb = pd.DataFrame({
    'Feature': X.columns,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 7))
palette_xgb = sns.color_palette('Oranges_r', len(fi_xgb))
ax.barh(fi_xgb['Feature'][::-1], fi_xgb['Importance'][::-1], color=palette_xgb)
ax.set_title('XGBoost — Top 20 Feature Importances (Gain)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Feature Importance Score')
plt.tight_layout()
plt.show()

---
## 7. SHAP Explainability (XGBoost)

In [ ]:
# ── 7.1 Compute SHAP Values ──
explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)
print(f'SHAP values shape: {shap_values.shape}')

In [ ]:
# ── 7.2 SHAP Summary Plot (Beeswarm) ──
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, plot_type='dot', max_display=20, show=False)
plt.title('SHAP Beeswarm — XGBoost Attrition Predictions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 7.3 SHAP Bar Plot (Mean |SHAP|) ──
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_test, plot_type='bar', max_display=20, show=False)
plt.title('Mean |SHAP| — Global Feature Importance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 7.4 SHAP Force Plot — Single High-Risk Employee ──
# Find the employee with the highest predicted probability
high_risk_idx = y_prob_xgb.argmax()

shap.initjs()
shap.force_plot(
    explainer.expected_value,
    shap_values[high_risk_idx],
    X_test.iloc[high_risk_idx],
    matplotlib=True
)
plt.title(f'SHAP Force Plot — Highest Risk Employee (P(leave)={y_prob_xgb[high_risk_idx]:.2%})',
          fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 7.5 SHAP Dependence Plots — Top 4 Drivers ──
top_features = fi_xgb['Feature'].head(4).tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    shap.dependence_plot(
        feat, shap_values, X_test,
        ax=axes[i], show=False,
        interaction_index='auto'
    )
    axes[i].set_title(f'SHAP Dependence: {feat}', fontweight='bold')

plt.suptitle('SHAP Dependence Plots — Top 4 Attrition Drivers', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## 8. Feature Importance — Side-by-Side Comparison

In [ ]:
# ── Normalise all importance scores to [0, 1] for fair comparison ──
top_n = 15

def top_features_df(series, name, n=top_n):
    s = series.abs() if name == 'Logistic Reg (|coef|)' else series
    s = s / s.max()        # normalise
    return s.nlargest(n).rename(name)

lr_fi  = top_features_df(pd.Series(lr.coef_[0], index=X.columns), 'Logistic Reg (|coef|)')
rf_fi  = top_features_df(pd.Series(rf.feature_importances_, index=X.columns), 'Random Forest')
xgb_fi = top_features_df(pd.Series(xgb_model.feature_importances_, index=X.columns), 'XGBoost')

shap_mean = pd.Series(np.abs(shap_values).mean(axis=0), index=X.columns)
shap_fi   = top_features_df(shap_mean, 'SHAP (XGBoost)')

all_top = set(lr_fi.index) | set(rf_fi.index) | set(xgb_fi.index) | set(shap_fi.index)
compare_df = pd.DataFrame(index=sorted(all_top))

for fi in [lr_fi, rf_fi, xgb_fi, shap_fi]:
    compare_df[fi.name] = fi

compare_df = compare_df.fillna(0)
compare_df['Mean'] = compare_df.mean(axis=1)
compare_df = compare_df.sort_values('Mean', ascending=False).head(top_n)

fig, ax = plt.subplots(figsize=(13, 8))
x = np.arange(len(compare_df))
width = 0.2
colors_fi = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']
cols_to_plot = ['Logistic Reg (|coef|)', 'Random Forest', 'XGBoost', 'SHAP (XGBoost)']

for i, (col, color) in enumerate(zip(cols_to_plot, colors_fi)):
    ax.bar(x + i * width, compare_df[col], width, label=col, color=color, alpha=0.85)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(compare_df.index, rotation=40, ha='right', fontsize=10)
ax.set_ylabel('Normalised Importance Score')
ax.set_title(f'Feature Importance Comparison — Top {top_n} Features\n'
             '(All scores normalised to [0,1] for comparability)',
             fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print('\nTop features across all methods:')
print(compare_df[cols_to_plot].round(3))

---
## 9. Model Comparison Summary

In [ ]:
def metrics_dict(name, y_true, y_pred, y_prob):
    return {
        'Model': name,
        'Accuracy':  accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall':    recall_score(y_true, y_pred),
        'F1':        f1_score(y_true, y_pred),
        'ROC-AUC':   roc_auc_score(y_true, y_prob)
    }

results = pd.DataFrame([
    metrics_dict('Logistic Regression', y_test, y_pred_lr,  y_prob_lr),
    metrics_dict('Random Forest',       y_test, y_pred_rf,  y_prob_rf),
    metrics_dict('XGBoost',             y_test, y_pred_xgb, y_prob_xgb),
]).set_index('Model').round(4)

print('=== Model Performance Summary ===')
print(results.to_string())
print('\n★ Best Recall (minimises missed leavers):', results['Recall'].idxmax())
print('★ Best ROC-AUC:', results['ROC-AUC'].idxmax())

In [ ]:
# ── Radar / Grouped Bar Comparison ──
metrics_cols = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
x_pos = np.arange(len(metrics_cols))
width = 0.25

fig, ax = plt.subplots(figsize=(13, 6))
model_colors = ['#2196F3', '#4CAF50', '#FF9800']

for i, (model, color) in enumerate(zip(results.index, model_colors)):
    vals = results.loc[model, metrics_cols].values
    bars = ax.bar(x_pos + i * width, vals, width, label=model, color=color, alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f'{v:.2f}', ha='center', va='bottom', fontsize=8.5)

ax.set_xticks(x_pos + width)
ax.set_xticklabels(metrics_cols, fontsize=12)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison\nLogistic Regression vs Random Forest vs XGBoost',
             fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── All ROC Curves Together ──
fig, ax = plt.subplots(figsize=(8, 6))

for (name, fpr, tpr, color) in [
    ('Logistic Regression', *roc_curve(y_test, y_prob_lr)[:2],  '#2196F3'),
    ('Random Forest',       *roc_curve(y_test, y_prob_rf)[:2],  '#4CAF50'),
    ('XGBoost',             *roc_curve(y_test, y_prob_xgb)[:2], '#FF9800'),
]:
    auc = roc_auc_score(y_test, {'Logistic Regression': y_prob_lr,
                                  'Random Forest': y_prob_rf,
                                  'XGBoost': y_prob_xgb}[name])
    ax.plot(fpr, tpr, lw=2, color=color, label=f'{name} (AUC={auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Baseline (AUC=0.500)')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — All Models', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

---
## 10. At-Risk Employee Scoring

In [ ]:
# ── Assign attrition probability to every employee in the test set ──
risk_df = X_test.copy()
risk_df['Actual_Attrition']  = y_test.values
risk_df['Risk_Score_XGB']    = y_prob_xgb
risk_df['Risk_Score_RF']     = y_prob_rf
risk_df['Predicted_Attrition'] = y_pred_xgb

risk_df['Risk_Tier'] = pd.cut(
    risk_df['Risk_Score_XGB'],
    bins=[0, 0.3, 0.6, 1.0],
    labels=['Low', 'Medium', 'High']
)

print('Risk Tier Distribution:')
print(risk_df['Risk_Tier'].value_counts())
print('\nTop 10 Highest-Risk Employees:')
risk_df.sort_values('Risk_Score_XGB', ascending=False).head(10)[
    ['Risk_Score_XGB', 'Risk_Score_RF', 'Risk_Tier', 'Actual_Attrition']
]

In [ ]:
# ── Risk Score Distribution Plot ──
fig, ax = plt.subplots(figsize=(10, 5))
stayed = risk_df[risk_df['Actual_Attrition'] == 0]['Risk_Score_XGB']
left   = risk_df[risk_df['Actual_Attrition'] == 1]['Risk_Score_XGB']

ax.hist(stayed, bins=30, alpha=0.65, color='#2196F3', label='Stayed (actual)')
ax.hist(left,   bins=30, alpha=0.65, color='#F44336', label='Left (actual)')
ax.axvline(0.3, color='orange', linestyle='--', linewidth=1.5, label='Medium threshold')
ax.axvline(0.6, color='red',    linestyle='--', linewidth=1.5, label='High threshold')
ax.set_xlabel('XGBoost Attrition Probability')
ax.set_ylabel('Employee Count')
ax.set_title('Risk Score Distribution — Test Set', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

---
## 11. Key Findings & Recommendations

### Key Findings
| Finding | Detail |
|---------|--------|
| **Overall Attrition** | 16.1% — 237 of 1,470 employees left |
| **Overtime** | 30.5% attrition vs 10.4% without — the single strongest predictor |
| **Sales Reps** | 39.8% attrition — 2.5× the company average |
| **Compensation gap** | Leavers earn ~$2,046/mo less than stayers |
| **Age risk** | 18–25 year-olds leave at 35.8% — nearly 4× the 36–45 cohort |
| **Best model** | XGBoost — AUC 0.74, Recall 61.7% on the held-out test set |

### Four Actionable Interventions
1. **Address Overtime Structurally** — Audit roles where OT is routine; cap monthly OT and track breach rates. Model's #1 predictor.
2. **Overhaul Sales Rep Retention** — Benchmark pay to market; build 6–18 month career tracks; introduce quarterly pulse surveys.
3. **Early-Career Engagement** — 12-month structured onboarding with mentoring and explicit career pathways for under-25s.
4. **Expand Equity & Benefits** — Extend stock option grants to front-line staff; 2-year vest to incentivise longer tenure.